In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.3-70b-versatile")

In [ ]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str = Field(description="Title of the movie")
    year:int = Field(description="Year of the movie")
    director:str = Field(description="Director of the movie")
    rating:float = Field(description="Rating of the movie out of 10")


In [ ]:
 model_with_struc = model.with_structured_output(Movie)
 model_with_struc.invoke("Provide details of spider man")

MESSAGE OUTPUT ALONGSIDE PARSED STRUCTURE

In [ ]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    """A Movie with detail"""
    title:str = Field(...,description="Title of the movie")
    year:int = Field(...,description="Year of the movie")
    director:str = Field(...,description="Director of the movie")
    rating:float = Field(...,description="Rating of the movie out of 10")
model_with_struc = model.with_structured_output(Movie,include_raw=True)
model_with_struc.invoke("Provide details of spider man")



NESTED STRUCTURE

In [ ]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None = Field(None,description="Budget in millions USD")


model_with_struc = model.with_structured_output(MovieDetails)
model_with_struc.invoke("Provide details of spider man")

TYPED DICT

In [6]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_with_type_dict=model.with_structured_output(MovieDict)
response=model_with_type_dict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avengers', 'year': 2012}

In [ ]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

In [ ]:
model.profile


DATA CLASSES

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

In [ ]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]